In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
### Check if OPENAI Client works

from openai import OpenAI
openai_client = OpenAI() 

In [4]:
# How the openai_client.responses.create() method works

llm_engine = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = "what is the capital of France?"
)

print(llm_engine.output_text)

The capital of France is **Paris**.


In [5]:
# wrap the openai_client.responses.create() method in a function

def llm(question):
    response = openai_client.responses.create(
        model = "gpt-5.4-mini",
        input = question
    )
    return response.output_text

In [6]:
# Ask  a generic question to the LLM and print the answer

question1 = "What is the capital of France?"
answer1 = llm(question1)
print(f"Question: {question1}")
print(f"Answer: {answer1}")

Question: What is the capital of France?
Answer: The capital of France is **Paris**.


In [7]:
#  Ask a domain specific question to the LLM and print the answer
question2  = 'When will the course be offered next?'    #"what products does RF Health produce"
answer2 = llm(question2)
print(f"Question2: {question2}")
print(f"Answer2: {answer2}")



Question2: When will the course be offered next?
Answer2: I can help, but I need the course name or details to find the next offering.

Please send one of these:
- the course title
- the institution/provider
- a link or screenshot of the course page

Then I can tell you when it’s next offered.


## -->  The generic llm struggles with domain specific answers so we need to feed or improve knowledge of this llm with the domain info or context.

In [8]:
context = """ 
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#How should I start the course and follow the weekly workflow?
Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

Watch the lesson videos.
Work through the lesson notebooks/code.
Read the homework instructions on GitHub.
Submit answers through the course platform before the deadline.
Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
If the homework title contains [DRAFT], it means the homework is not ready yet.

The homework is ready only when both are true:

The homework form is open on the course management platform.
The homework title does not contain [DRAFT].
Until then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.

edit on GitHub
#When will the course be offered next?
Summer 2027.
"""

In [9]:

#question = "Can I submit homework after the deadline, or get a deadline extension?"
question =  'When will the course be offered next?'


prompt = f'''
Your task is to answer questions based on the provided context. 
Use the information in the context to generate accurate, relevant and helpful responses.
If the answer is not explicitly stated in the context, provide a well-reasoned answer based on the information available.
If the answer cannot be determined from the context, respond with "I don't know."

Question: 
{question}

Context: 
{context}
'''

In [10]:
print(prompt)


Your task is to answer questions based on the provided context. 
Use the information in the context to generate accurate, relevant and helpful responses.
If the answer is not explicitly stated in the context, provide a well-reasoned answer based on the information available.
If the answer cannot be determined from the context, respond with "I don't know."

Question: 
When will the course be offered next?

Context: 
 
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link

In [11]:
#  Ask a domain specific question to the LLM and print the answer
question  = 'When will the course be offered next?'   
answer = llm(prompt)
print(f"Question: {question}")
print(f"Answer: {answer}")



Question: When will the course be offered next?
Answer: Summer 2027.


## RAG -  Retrieval Augmented Generation
# Datasets

In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)    
    return llm(user_prompt)

In [15]:
#--> This returns a list of courses. Each course has a path field that points to its FAQ data.

import requests
docs_url = 'https://datatalks.club/faq/json/courses.json'  #index of all courses and their FAQs
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw


[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 103},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [29]:
#--> Fetch all FAQ documents from all courses
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

print(len(documents))  # Check the number of documents fetched
documents[5]

1368


{'id': 'b71fb3b195',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: how many Zoomcamps run in a year?',
 'answer': 'DataTalks.Club runs several Zoomcamps every year. The roster and approximate timing:\n\n- Data Engineering: Jan – Apr\n- Stock Market Analytics: Apr – May\n- MLOps: May – Aug\n- LLM: Jun – Sep\n- Machine Learning: Sep – Jan\n\nFor the up-to-date list and current dates, see the [DataTalks.Club guide to free courses](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\nEach Zoomcamp has one "live" cohort per year — that\'s the only window in which you can earn the certificate. Outside the live cohort you can still take the course self-paced (materials stay open), but no certificate.'}

## Search --> indexing data with minsearch

In [21]:
from minsearch import Index

index = Index(
    text_fields=['section','question','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [45]:
index.search(question, 
             filter_dict={'course': 'llm-zoomcamp'},
             num_results=3
             )

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '651ba06b34',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Will the name I put in the certificate field be shown publicly online or shared with third parties?',
  'answer': "No. The certificate name only appears on your certificate — it isn't published online or shared with third parties. The public leaderboard uses your separate display name instead, which is a random nickname (like “Lucid Elbakyan”) and anonymous by default.\n\nYou can set both in “Edit Course Profile”: the first field is your public nickname for the leaderboard, and the second is the official name printed on your certificate. This lets you keep your real name off the public leaderboard while still having it on the certificate."},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  '

In [ ]:
# boosting
index.search(
    query=question,
    boost_dict={'question':2, 'section':0.5},  #question has level of importance 2, section has level of importance 0.5
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=3
)

123


In [ ]:
# By Default, it searches the LLM Zoomcamp FAQ

def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        query= 'when can i join the course',    # ---> question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=3
    )     

search_results = search(question)
search_results

# #All fields have a default boost of 1. Giving question a boost of 2 means it counts two times as much. Take a question about certificates. The word "certificate" in the question field now weighs twice what it does in the answer.

# Giving section 0.5 means it counts half as much, since a match there tells us less. This is the same boosting mechanism used by Elasticsearch and Lucene.

# Try different boost values and see how the results change.

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'}]

### Building Prompt